# FinBERT fine-tune (Kaggle GPU)

Runs `pipeline/kaggle_finbert_train.py` on this notebook's GPU. The training
set is built locally by `pipeline/export_training_set.py`; this notebook only
consumes its three CSVs, so nothing here re-implements a filter or a split.

**Read the two metric blocks it prints as different things.** `val` labels come
from the labeler, so val measures agreement with Llama, not accuracy. `holdout`
labels are human and were kept out of training — only that block says whether
the model is right.

**And do not read its `negative` numbers as evidence about distress
detection.** The training set carries 38 `negative` rows and the holdout 6;
neither can support that claim. See `scoring/DESIGN.md` decision log
2026-08-09.

## Prepare the upload (run this locally, in the repo root)

```sh
rm -rf kaggle_upload
mkdir -p kaggle_upload/pipeline
cp pipeline/__init__.py pipeline/labeling.py \
   pipeline/kaggle_finbert_train.py  kaggle_upload/pipeline/
cp finbert_<date>_train.csv finbert_<date>_val.csv \
   finbert_<date>_holdout.csv        kaggle_upload/
```

Upload `kaggle_upload/` as a **private** Kaggle dataset — the training CSVs are
gitignored data and must not become public.

## Then, in this notebook

1. Settings → Accelerator → **GPU** (T4 x2 or P100).
2. Settings → Internet → **On** (the checkpoint download needs it).
3. Add Data → your private dataset.
4. Point `UPLOAD_DIR` below at it, leave `SMOKE_TEST = True`, and Run All.
5. If the smoke run finishes clean, set `SMOKE_TEST = False` and Run All again.

The smoke pass exists because this script has never run: torch and
transformers are not installed in the repo's environment, so several lines
marked `calibration` — the constrained `compute_loss` signature, the
checkpoint's `id2label` order — are unverified until a GPU sees them. Better
to find that out in one minute than twenty.

In [ ]:
# --- config: the only cell you normally edit -------------------------------

# The attached dataset. Copy the exact path from the right-hand panel.
UPLOAD_DIR = "/kaggle/input/pnc-finbert"

PREFIX = "finbert_2026-08-09"  # the export_training_set --output-prefix

# First run: 200/100 rows, 1 epoch, just to prove the script executes.
SMOKE_TEST = True

EPOCHS = 3
OUTPUT_DIR = "/kaggle/working/finbert_ft"
WORK_DIR = "/kaggle/working/run"

In [ ]:
import csv
import os
import shutil
from collections import Counter
from datetime import UTC, datetime

assert os.path.isdir(UPLOAD_DIR), (
    f"{UPLOAD_DIR} not found — fix UPLOAD_DIR. Attached: "
    f"{os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'nothing'}"
)

# Re-runnable, and /kaggle/input is read-only while the package must be
# importable as `pipeline.` from the working directory.
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(UPLOAD_DIR, WORK_DIR)
os.chdir(WORK_DIR)

SPLITS = {s: f"{PREFIX}_{s}.csv" for s in ("train", "val", "holdout")}
REQUIRED = [
    "pipeline/__init__.py",
    "pipeline/labeling.py",
    "pipeline/kaggle_finbert_train.py",
    *SPLITS.values(),
]
absent = [p for p in REQUIRED if not os.path.exists(p)]
assert not absent, f"missing from the upload: {absent}"

RUN_DATE = datetime.now(UTC).date().isoformat()
print("staged  :", WORK_DIR)
print("run date:", RUN_DATE)

In [ ]:
# Fail here, before the checkpoint download, rather than after it.
from pipeline.labeling import LABELS

for split, path in SPLITS.items():
    with open(path, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    missing = {"raw_item_id", "text", "label"} - set(rows[0])
    assert not missing, f"{path} is missing columns: {missing}"
    bad = {r["label"] for r in rows} - set(LABELS)
    assert not bad, f"{path} has invalid labels: {bad}"
    assert all(r["text"].strip() for r in rows), f"{path} has an empty text field"
    print(f"{split:8} {len(rows):>5}  {dict(Counter(r['label'] for r in rows))}")

# The training set and the holdout must not share rows, or the holdout is
# measuring memorisation. export_training_set separates them; verify anyway,
# because this is the one property the whole holdout argument rests on.
ids = {
    s: {r["raw_item_id"] for r in csv.DictReader(open(p, encoding="utf-8"))}
    for s, p in SPLITS.items()
}
assert not ids["train"] & ids["holdout"], "holdout rows leaked into train"
assert not ids["val"] & ids["holdout"], "holdout rows leaked into val"
assert not ids["train"] & ids["val"], "train and val overlap"
print("\nno overlap between train, val and holdout")

In [ ]:
# Kaggle ships torch and transformers; print what this run actually got, so a
# version-sensitive failure below can be traced to a version.
import torch
import transformers

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print(
    "cuda        ",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
)

In [ ]:
# Build the run's arguments: the smoke pass trims the splits and the epochs.
run_splits, epochs, out_dir = dict(SPLITS), EPOCHS, OUTPUT_DIR
if SMOKE_TEST:
    epochs, out_dir = 1, OUTPUT_DIR + "_smoke"
    for split, n in (("train", 200), ("val", 100), ("holdout", 50)):
        with open(SPLITS[split], encoding="utf-8") as f:
            rows = list(csv.DictReader(f))
        # Sample per class rather than slicing the head. Head rows are almost
        # all neutral, and taking directional rows first would leave train
        # with no neutral at all -- either way a class the run never sees.
        keep = []
        for label in LABELS:
            keep += [r for r in rows if r["label"] == label][: max(1, n // len(LABELS))]
        run_splits[split] = f"smoke_{split}.csv"
        with open(run_splits[split], "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["raw_item_id", "text", "label"])
            w.writeheader()
            w.writerows(keep)
        counts = dict(Counter(r["label"] for r in keep))
        assert len(counts) == len(LABELS), f"smoke {split} missing a class: {counts}"
        print(f"smoke {split:8} {len(keep):>4}  {counts}")

TRAIN, VAL, HOLDOUT = run_splits["train"], run_splits["val"], run_splits["holdout"]
EPOCHS_ARG, OUT = str(epochs), out_dir
mode = "SMOKE" if SMOKE_TEST else "FULL"
print(f"\nmode: {mode} | epochs: {epochs} | -> {out_dir}")

In [ ]:
!python -m pipeline.kaggle_finbert_train \
    --train "$TRAIN" \
    --val "$VAL" \
    --holdout "$HOLDOUT" \
    --output-dir "$OUT" \
    --run-date "$RUN_DATE" \
    --epochs "$EPOCHS_ARG"

In [ ]:
# What the run actually produced. On a smoke pass the numbers mean nothing —
# only that the script ran end to end and wrote its artefacts.
import json

with open(os.path.join(out_dir, "metrics.json"), encoding="utf-8") as f:
    m = json.load(f)

print(
    m["model_version"],
    "| train",
    m["train_size"],
    "| val",
    m["val_size"],
    "| holdout",
    m["holdout_size"],
)
for block, title in (
    ("", "val — agreement with the LABELER"),
    ("_holdout", "holdout — human truth"),
):
    before, after = m.get("baseline_pretrained" + block), m.get("fine_tuned" + block)
    if not before:
        continue
    print(f"\n{title}")
    print(f"{'':10}{'pretrained':>12}{'fine-tuned':>12}")
    print(f"{'accuracy':10}{before['accuracy']:>12.3f}{after['accuracy']:>12.3f}")
    for lb in ("positive", "negative", "neutral"):
        print(f"f1 {lb:7}{before[lb]['f1']:>12.3f}{after[lb]['f1']:>12.3f}")

print("\nweights:", sorted(os.listdir(out_dir)))

## Next

Once the full run looks sane, download `/kaggle/working/finbert_ft` and publish
it as a **private Kaggle dataset** — weights are versioned outside the repo
(`scoring/DESIGN.md` Stage 2). Record the `model_version` string from
`metrics.json`; `item_score.model_version` names it for every row the model
scores.

Keep `metrics.json` with the weights. The pretrained-vs-fine-tuned comparison
in it is the only evidence the fine-tune did anything, and the write-up needs
the holdout block specifically — not the val block, whose labels came from the
labeler being distilled.